# AffineTensor tests

Regression tests for `intervalnets.affine.AffineTensor`.

In [ ]:
from pathlib import Path
import sys

_cwd = Path.cwd().resolve()
_src = _cwd / 'src'
if not _src.exists():
    _src = _cwd.parent / 'src'
sys.path.insert(0, str(_src.resolve()))

from math import inf

import pytest

from intervalnets.affine import AffineTensor


In [ ]:
# test_point_has_zero_generators_and_exact_bounds
value = [1.0, -2.5]
affine = AffineTensor.point(value)
lower, upper = affine.to_bounds()
assert affine.c == (1.0, -2.5)
assert affine.G == ((), ())
assert lower[0] <= 1.0 <= upper[0]
assert lower[1] <= -2.5 <= upper[1]

In [ ]:
# test_from_bounds_round_trips_interval_conservatively
affine = AffineTensor.from_bounds([-1.0, 2.0], [3.0, 5.0])
lower, upper = affine.to_bounds()
assert lower[0] <= -1.0
assert upper[0] >= 3.0
assert lower[1] <= 2.0
assert upper[1] >= 5.0

In [ ]:
# test_affine_add_sub_and_negation_preserve_enclosure
a = AffineTensor.from_bounds([0.0, -1.0], [1.0, 2.0])
b = AffineTensor.from_bounds([-2.0, 1.0], [0.5, 3.0])
c = a + b
d = a - b
e = -a
c_lower, c_upper = c.to_bounds()
d_lower, d_upper = d.to_bounds()
e_lower, e_upper = e.to_bounds()
assert c_lower[0] <= -2.0 and c_upper[0] >= 1.5
assert c_lower[1] <= 0.0 and c_upper[1] >= 5.0
assert d_lower[0] <= -0.5 and d_upper[0] >= 3.0
assert d_lower[1] <= -4.0 and d_upper[1] >= 1.0
assert e_lower[0] <= -1.0 and e_upper[0] >= 0.0
assert e_lower[1] <= -2.0 and e_upper[1] >= 1.0

In [ ]:
# test_affine_scalar_arithmetic
a = AffineTensor.from_bounds([1.0, 2.0], [2.0, 4.0])
plus = a + 3.0
minus = 10.0 - a
plus_lower, plus_upper = plus.to_bounds()
minus_lower, minus_upper = minus.to_bounds()
assert plus_lower[0] <= 4.0 and plus_upper[0] >= 5.0
assert plus_lower[1] <= 5.0 and plus_upper[1] >= 7.0
assert minus_lower[0] <= 8.0 and minus_upper[0] >= 9.0
assert minus_lower[1] <= 6.0 and minus_upper[1] >= 8.0

In [ ]:
# test_affine_map_matches_matrix_rule_on_center_and_generators
z = AffineTensor.from_bounds([-1.0, 0.0], [3.0, 2.0])
W = ((2.0, -1.0), (0.5, 3.0))
b = (0.25, -2.0)
mapped = z.affine_map(W, b)
expected_center = (2.0 * z.c[0] - 1.0 * z.c[1] + 0.25, 0.5 * z.c[0] + 3.0 * z.c[1] - 2.0)
assert mapped.c == pytest.approx(expected_center)
lower, upper = mapped.to_bounds()
corners = [
    (2.0 * x - 1.0 * y + 0.25, 0.5 * x + 3.0 * y - 2.0)
    for x in (-1.0, 3.0)
    for y in (0.0, 2.0)
]
assert lower[0] <= min(c[0] for c in corners)
assert upper[0] >= max(c[0] for c in corners)
assert lower[1] <= min(c[1] for c in corners)
assert upper[1] >= max(c[1] for c in corners)

In [ ]:
# test_affine_map_dimension_validation
z = AffineTensor.from_bounds([-1.0, 0.0], [3.0, 2.0])
with pytest.raises(ValueError, match='Dimension mismatch'):
    _ = z.affine_map(((1.0, 2.0, 3.0),), (0.0,))

In [ ]:
# test_to_bounds_is_outward_rounded
z = AffineTensor.from_bounds(0.0, 1.0)
lower, upper = z.to_bounds()
assert lower < 0.0
assert upper > 1.0
assert lower != -inf and upper != inf

In [ ]:
# torch-backed affine activation transform setup
import torch

from intervalnets.affine_pytorch import (
    affine_relu_transform,
    affine_sigmoid_transform,
    affine_tanh_transform,
)


In [ ]:
# test_affine_activation_transforms_enclose_samples
def _assert_encloses_samples(transform_fn, point_fn, lower, upper, samples: int = 2000):
    x = AffineTensor.from_bounds(torch.tensor(lower, dtype=torch.float64), torch.tensor(upper, dtype=torch.float64))
    y = transform_fn(x)
    y_lower, y_upper = y.to_bounds()

    rand = torch.rand(samples, len(lower), dtype=torch.float64)
    lo = torch.tensor(lower, dtype=torch.float64)
    hi = torch.tensor(upper, dtype=torch.float64)
    xs = lo + (hi - lo) * rand
    ys = point_fn(xs)

    assert torch.all(ys >= y_lower.unsqueeze(0))
    assert torch.all(ys <= y_upper.unsqueeze(0))


_assert_encloses_samples(
    affine_relu_transform,
    lambda x: torch.relu(x),
    lower=[-2.0, -1.0, 0.2],
    upper=[3.0, 2.5, 1.4],
)

_assert_encloses_samples(
    affine_tanh_transform,
    lambda x: torch.tanh(x),
    lower=[-2.5, -0.5, 0.0],
    upper=[1.5, 2.0, 3.0],
)

_assert_encloses_samples(
    affine_sigmoid_transform,
    lambda x: torch.sigmoid(x),
    lower=[-6.0, -1.0, 0.2],
    upper=[-2.0, 3.0, 4.0],
)


In [ ]:
# test_affine_activation_transforms_handle_degenerate_intervals_exactly
point = torch.tensor([0.0, -1.5, 2.0], dtype=torch.float64)
x = AffineTensor.point(point)

relu_out = affine_relu_transform(x)
tanh_out = affine_tanh_transform(x)
sigmoid_out = affine_sigmoid_transform(x)

assert torch.allclose(relu_out.c, torch.relu(point))
assert torch.allclose(tanh_out.c, torch.tanh(point))
assert torch.allclose(sigmoid_out.c, torch.sigmoid(point))

zeros = torch.zeros(point.numel(), point.numel(), dtype=torch.float64)
assert torch.allclose(relu_out.G[:, -point.numel() :], zeros)
assert torch.allclose(tanh_out.G[:, -point.numel() :], zeros)
assert torch.allclose(sigmoid_out.G[:, -point.numel() :], zeros)


In [ ]:
# test_pytorch_domain_dispatch_with_affine_inputs
from intervalnets.pytorch import enable_interval_eval, interval_forward

linear_relu = torch.nn.Sequential(torch.nn.Linear(2, 2), torch.nn.ReLU())
with torch.no_grad():
    linear_relu[0].weight.copy_(torch.tensor([[1.0, -1.0], [0.25, 0.5]], dtype=torch.float32))
    linear_relu[0].bias.copy_(torch.tensor([0.0, 0.1], dtype=torch.float32))

affine_domain = AffineTensor.from_bounds(
    torch.tensor([-1.0, 0.0], dtype=torch.float32),
    torch.tensor([1.0, 1.5], dtype=torch.float32),
)

forward_out = interval_forward(linear_relu, affine_domain)
assert isinstance(forward_out, AffineTensor)

enable_interval_eval()
eval_out = linear_relu.eval(affine_domain)
assert isinstance(eval_out, AffineTensor)

jacobian_out = linear_relu.eval_jacobian(affine_domain)
hessian_out = linear_relu.eval_hessian(affine_domain)
assert jacobian_out.lower[0][0] <= jacobian_out.upper[0][0]
assert jacobian_out.lower[0][1] <= jacobian_out.upper[0][1]
assert hessian_out.lower[0][0][0] <= hessian_out.upper[0][0][0]
assert hessian_out.lower[0][1][1] <= hessian_out.upper[0][1][1]



In [ ]:
# additional parity checks for affine unit tests added in tests/
from intervalnets import IntervalTensor, affine_forward

# affine_map torch semantics: Wc+b and WG
x = AffineTensor.from_bounds(
    torch.tensor([-1.0, 2.0], dtype=torch.float64),
    torch.tensor([3.0, 4.0], dtype=torch.float64),
)
W = torch.tensor([[2.0, -1.0], [0.5, 3.0]], dtype=torch.float64)
b = torch.tensor([0.25, -0.75], dtype=torch.float64)
mapped = x.affine_map(W, b)
assert torch.allclose(mapped.c, W @ x.c + b)
assert torch.allclose(mapped.G, W @ x.G)

# interval_forward/model.eval compatibility with both interval and affine domains
enable_interval_eval()
model = torch.nn.Sequential(torch.nn.Linear(2, 2), torch.nn.ReLU())
with torch.no_grad():
    model[0].weight.copy_(torch.tensor([[1.0, -0.5], [0.5, 2.0]], dtype=torch.float32))
    model[0].bias.copy_(torch.tensor([0.0, 0.2], dtype=torch.float32))

interval_domain = IntervalTensor.from_bounds([-1.0, 0.0], [1.0, 2.0])
affine_domain = AffineTensor.from_bounds(
    torch.tensor([-1.0, 0.0], dtype=torch.float32),
    torch.tensor([1.0, 2.0], dtype=torch.float32),
)

interval_out = interval_forward(model, interval_domain)
affine_out = interval_forward(model, affine_domain)
eval_interval_out = model.eval(interval_domain)
eval_affine_out = model.eval(affine_domain)

assert isinstance(interval_out, IntervalTensor)
assert isinstance(affine_out, AffineTensor)
assert isinstance(eval_interval_out, IntervalTensor)
assert isinstance(eval_affine_out, AffineTensor)

# affine_forward helper dispatch
affine_out_via_helper = affine_forward(model, affine_domain)
assert isinstance(affine_out_via_helper, AffineTensor)


# affine_forward supports torch sequential linear+tanh and fallback linear backend
backend_model = torch.nn.Sequential(torch.nn.Linear(2, 2), torch.nn.Tanh())
with torch.no_grad():
    backend_model[0].weight.copy_(torch.tensor([[1.5, -0.5], [0.25, 2.0]], dtype=torch.float64))
    backend_model[0].bias.copy_(torch.tensor([0.1, -0.2], dtype=torch.float64))

torch_backend_domain = AffineTensor.from_bounds(
    torch.tensor([-1.0, 0.25], dtype=torch.float64),
    torch.tensor([0.5, 1.75], dtype=torch.float64),
)
torch_backend_out = affine_forward(backend_model, torch_backend_domain)
torch_backend_lower, torch_backend_upper = torch_backend_out.to_bounds()
assert torch.all(torch_backend_lower <= torch_backend_upper)

fallback_backend_domain = AffineTensor.from_bounds(tuple([-1.0, 0.25]), tuple([0.5, 1.75]))
fallback_backend_out = affine_forward(backend_model[0], fallback_backend_domain)
fallback_backend_lower, fallback_backend_upper = fallback_backend_out.to_bounds()
assert all(lower <= upper for lower, upper in zip(fallback_backend_lower, fallback_backend_upper))


In [ ]:
# affine interval-eval extensions for lpnorm/sobolev/jacobian/hessian on affine domains
from intervalnets import AffineTensor, enable_interval_eval

enable_interval_eval()
model = torch.nn.Sequential(torch.nn.Linear(2, 3), torch.nn.Tanh(), torch.nn.Linear(3, 1))
with torch.no_grad():
    model[0].weight.copy_(torch.tensor([[0.8, -0.4], [0.3, 0.5], [-0.7, 0.2]], dtype=torch.float32))
    model[0].bias.copy_(torch.tensor([0.1, -0.2, 0.05], dtype=torch.float32))
    model[2].weight.copy_(torch.tensor([[1.1, -0.3, 0.6]], dtype=torch.float32))
    model[2].bias.copy_(torch.tensor([0.0], dtype=torch.float32))

domain = AffineTensor.from_bounds(
    torch.tensor([-0.5, -0.25], dtype=torch.float32),
    torch.tensor([0.5, 0.75], dtype=torch.float32),
)

lp = model.lpnorm(domain, p=2.0, iterations=1)
jacobian = model.eval_jacobian(domain)
hessian = model.eval_hessian(domain)
sobolev = model.sobolev_norm(domain, p=2.0, order=1, iterations=1)

assert math.isfinite(float(lp.lower)) and math.isfinite(float(lp.upper))
assert float(lp.lower) <= float(lp.upper)
assert math.isfinite(float(sobolev.lower)) and math.isfinite(float(sobolev.upper))
assert float(sobolev.lower) <= float(sobolev.upper)
assert jacobian.lower[0][0] <= jacobian.upper[0][0]
assert jacobian.lower[0][1] <= jacobian.upper[0][1]
assert hessian.lower[0][0][0] <= hessian.upper[0][0][0]
assert hessian.lower[0][0][1] <= hessian.upper[0][0][1]
assert hessian.lower[0][1][0] <= hessian.upper[0][1][0]
assert hessian.lower[0][1][1] <= hessian.upper[0][1][1]

torch.manual_seed(13)
mc_model = torch.nn.Sequential(torch.nn.Linear(2, 4), torch.nn.Tanh(), torch.nn.Linear(4, 1))
with torch.no_grad():
    for parameter in mc_model.parameters():
        torch.nn.init.uniform_(parameter, a=-0.7, b=0.7)

lower = torch.tensor([-0.4, -0.2], dtype=torch.float32)
upper = torch.tensor([0.6, 0.5], dtype=torch.float32)
mc_domain = AffineTensor.from_bounds(lower, upper)
lp_bounds = mc_model.lpnorm(mc_domain, p=2.0, iterations=2)
sobolev_bounds = mc_model.sobolev_norm(mc_domain, p=2.0, order=1, iterations=2)

samples = torch.rand(10000, 2, dtype=torch.float64)
samples[:, 0] = samples[:, 0] * float(upper[0] - lower[0]) + float(lower[0])
samples[:, 1] = samples[:, 1] * float(upper[1] - lower[1]) + float(lower[1])
values = mc_model(samples.to(dtype=torch.float32)).to(dtype=torch.float64).squeeze(-1)
volume = float((upper[0] - lower[0]) * (upper[1] - lower[1]))
lp_estimate = (volume * torch.mean(values.abs().pow(2.0)).item()) ** 0.5
assert float(lp_bounds.lower) <= lp_estimate <= float(lp_bounds.upper)

gradients = []
for sample in samples[:512]:
    x = sample.to(dtype=torch.float32).clone().detach().requires_grad_(True)
    y = mc_model(x.unsqueeze(0)).squeeze()
    grad = torch.autograd.grad(y, x, create_graph=False)[0].to(dtype=torch.float64)
    gradients.append(float(torch.sum(grad * grad).item()))
grad_sq_mean = sum(gradients) / len(gradients)
sobolev_integrand_estimate = torch.mean(values.abs().pow(2.0)).item() + grad_sq_mean
sobolev_estimate = (volume * sobolev_integrand_estimate) ** 0.5
assert float(sobolev_bounds.lower) <= sobolev_estimate <= float(sobolev_bounds.upper)

